# Medical Chatbot

## Import FAISS Index and RAG text

In [1]:
!pip install faiss-cpu
!pip install -U transformers>=4.44.0 accelerate
!pip install -U bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 57.5 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.1.1 requires pyarrow>=21.0.0, but you have pyarrow 19.0.1 which is incompatible.
libcugraph-cu12 25.6.0 requires libraft-cu12==25.6.*, but you have libraft-cu12 25.2.0 which is incompatible.
gradio 5.38.1 requires pydantic<2.12,>=2.0, but you have pydantic 2.12.0a1 which is incompatible.
pylibcugraph-cu12 25.6.0 requires pylibraft-cu12==25.6.*, but you have pylibraft-cu12 25.2.0 which is incompatible.
pylibcugraph-cu12 25.6.0 requires rmm-cu12==25.6.*, but you have rmm-cu12 25.2.0 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 17.8 MB/s eta 0:00:00:00:0100:01


In [2]:
import pickle   
import faiss        
with open("/kaggle/input/faiss/other/default/1/faiss_texts.pkl", "rb") as f:
    texts = pickle.load(f)     

index = faiss.read_index("/kaggle/input/faiss/other/default/1/faiss_index.index")   

In [3]:
from tqdm import tqdm
import torch
import numpy as np    
import faiss
from sentence_transformers import SentenceTransformer
from typing import List, Tuple

# Query Function for RAG

def query_rag(
    query: str,
    model: SentenceTransformer,
    index,
    texts: List[str],
    device: str = "cuda" if torch.cuda.is_available() else "cpu",
    top_k: int = 5,
    show_results: bool = True
):

    # Encode the query 
    query_vec = model.encode([query], convert_to_numpy=True, device=device)

    # Ensure correct dtype and shape
    query_vec = np.array(query_vec, dtype=np.float32)
    if len(query_vec.shape) == 1:
        query_vec = query_vec.reshape(1, -1)

    # Search in FAISS
    D, I = index.search(query_vec, top_k)

    # Retrieve text results
    contexts = [texts[i] for i in I[0]]
    scores = [float(s) for s in D[0]]

    return contexts


device = 'cuda' if torch.cuda.is_available() else 'cpu'
embedding_model = SentenceTransformer('Qwen/Qwen3-Embedding-0.6B')

2025-11-10 18:14:28.978095: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1762798469.151208      39 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1762798469.203410      39 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

## Import Pre-trained medical chatbot

In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "microsoft/MediPhi-Clinical"

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    load_in_4bit=True,
    trust_remote_code=True
)


# Create generation pipeline
from transformers import pipeline
medical_assistant = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device_map="auto"
)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/2.67G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/205 [00:00<?, ?B/s]

Device set to use cuda:0


In [5]:
# This pipeline will act as our smart router for both intent and seriousness
classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

print("Zero-shot classifier loaded successfully.")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


Zero-shot classifier loaded successfully.


## Create function to query the model

## Model 1

In [6]:
# generate_answer function

from pydantic import BaseModel
from typing import List

# We define this here so the function knows what a ChatMessage is
class ChatMessage(BaseModel):
    role: str
    content: str

# this function will only get called if the question is medical related
def generate_answer(query: str, context: list, history: List[ChatMessage], tokens: int = 1000) -> str:
    """
    Generates an answer using the LLM based on a query, context, AND conversation history.
    """
    
    # Format the history for the prompt
    formatted_history = ""
    for message in history:
        if message.role == 'user':
            formatted_history += f"Previous Question: {message.content}\n"
        elif message.role == 'assistant':
            formatted_history += f"Your Previous Answer: {message.content}\n---\n"

    # Build the prompt with the new history section
    system_prompt = f"""
You are a reliable, calm, and knowledgeable medical first aid assistant. You can remember the past conversation.
 
Your job is to give a **complete, step-by-step guide**.
 
Follow these strict rules:
- Use clear, simple, reassuring language.
- Avoid generic bullet lists — instead, structure the response in short sections.
- Highlight urgent actions with **bold** text.
- **If the query describes or implies an emergency / injury / accident / sudden illness / life-threatening scenario → switch 
to FIRST AID MODE.**
 
- **If the query is informational, preventive, or explanatory (e.g., about conditions, symptoms, nutrition, health advice) → 
switch to GENERAL MEDICAL MODE.**
- In case of emergencies, instruct the user to contact their *local* emergency services or visit the nearest medical facility.
- If the retrieved context mentions a specific emergency hotline (e.g., 911, 000, 112), replace it with the phrase 
‘your local emergency number’. In case a country is mentioned do not include it in the reponse.
- Do not mention or speculate about the user's country, location, or local hotlines — use only "your local emergency 
number" when advising to call emergency services.
- All words must be spelled correctly in standard English; do not produce garbled or partial words (e.g., “Immedi000”). If unsure, rewrite the phrase normally.
- If a protocol (like FAST or DRSABCD) appears, **expand and explain each letter.**
 
 
### FIRST AID MODE (for emergencies)
 
**Format:**
 
**Situation:** short summary of the problem  
Start with a calm reassurance: “Stay calm — quick action can make a big difference.”  
 
Then follow this exact structure:
 
1. **Immediate Actions** — 3–6 numbered steps showing exactly what to do first  
2. **Rationale** — one short paragraph explaining *why* those steps matter  
3. **What NOT to do** — 2–4 clear things to avoid  
4. **🚨 When to Seek Immediate Help** — short list of red-flag signs then append `<END>` and nothing after it
 
**Rules:**
- Highlight urgent actions with **bold**.  
- Always replace emergency numbers with “your local emergency number”.  
- If a first aid protocol (like DRSABCD or FAST) appears, **expand and explain each step clearly**.
 
---
 
### GENERAL MEDICAL MODE (for non-emergencies)
 
**Format:**
 
Then use this structure:
1. **Overview** — simple explanation of the concept or condition  
2. **Causes / Mechanism** — brief summary of underlying reason(s)  
3. **Common Symptoms or Signs** — bullet points if relevant  
4. **Management / Prevention** — clear, general guidance (no prescription-only details)  
5. **When to See a Doctor** — short note for red flags then append `<END>` and nothing after it. 

---

### Chat History:
{formatted_history}

### Context for the new question:
{context}

### Current Question:
{query}

Now provide your single response below:
"""

    # Generate
    inputs = tokenizer(system_prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=tokens,
            max_length = 1000,
            temperature=0.4,
            top_p=0.9,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )

    text = tokenizer.decode(output[0], skip_special_tokens=True)

    # Clean the response
    response = text.split("Now provide your single response below:")[-1]
    response = response.split("<END>")[0].strip()

    return response

# Classify as Medical or Non-Medical

In [7]:
from transformers import pipeline
from typing import List
from pydantic import BaseModel

# Define ChatMessage model for history tracking
class ChatMessage(BaseModel):
    role: str
    content: str

# Load the zero-shot classifier once
print("⏳ Loading BART-large-MNLI zero-shot model...")
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
print("✅ Model loaded successfully!")

# Check if query is medical or not ---
def is_medical_query(query: str) -> bool:
    """
    Uses zero-shot classification to detect whether a user query is medical-related.
    """
    result = classifier(query, candidate_labels=["medical", "non-medical"])
    top_label = result["labels"][0]
    top_score = result["scores"][0]
    print(f"🩺 Query classification → {top_label} ({top_score:.2f})")

    return top_label == "medical"

# Advanced intent classifier (uses history) ---

def classify_intent_advanced(query: str, history: List[ChatMessage]) -> str:
    """
    Classifies every query on its own merit using a calibrated threshold.
    This is the safest and most predictable approach.
    """
    query_lower = query.lower().strip()
    greetings = ["hello", "hi", "hey", "good morning", "good afternoon", "good evening"]

    # 1. Handle simple greetings first.
    if any(greeting in query_lower for greeting in greetings):
        return "greeting"

    # 2. Always classify the new query.
    candidate_labels = ["medical first-aid question", "general conversation"]
    result = classifier(query, candidate_labels)
    top_label = result['labels'][0]
    top_score = result['scores'][0]
    
    # 3. Use a simple, clear threshold to decide.
    # We found that 0.55 is a good balance.
    if top_label == "medical first-aid question" and top_score > 0.55:
        print(f"🧠 Intent classified as MEDICAL ({top_score:.2f})")
        return "medical"
    else:
        print(f"🧠 Intent classified as OFF_TOPIC ({top_score:.2f})")
        return "off_topic"

# --- 3️⃣ FUNCTION: Determine seriousness of response ---
def classify_response_seriousness(response_text: str) -> bool:
    """
    Detects whether the assistant's generated medical response
    describes a serious or life-threatening emergency.
    Returns True if it's serious.
    """
    candidate_labels = ["urgent medical emergency", "minor first-aid advice"]
    result = classifier(response_text, candidate_labels)

    top_label = result['labels'][0]
    top_score = result['scores'][0]

    print(f"⚕️ Seriousness check → {top_label} ({top_score:.2f})")

    return top_label == "urgent medical emergency" and top_score > 0.6


⏳ Loading BART-large-MNLI zero-shot model...


Device set to use cuda:0


✅ Model loaded successfully!


In [8]:


# This cell allows you to test ONLY the seriousness classification function.

from IPython.display import display, Markdown

# SET YOUR TEST RESPONSE TEXT HERE ---
# Write a sentence that LOOKS LIKE an answer the AI would give.

# Example of a response for a SERIOUS case (should return True)
serious_response_example = """
Based on the symptoms of sudden weakness on one side of the face, difficulty speaking, and confusion, this could be a stroke, which is a critical medical emergency. **You must call your local emergency services immediately.** While waiting for help, ensure the person is in a comfortable position and do not give them anything to eat or drink.
"""

# Example of a response for a MINOR case (should return False)
# minor_response_example = """
# For a small cut, the first step is to clean the wound gently with soap and water. Apply a sterile bandage to protect it from bacteria. Monitor the area for signs of infection such as redness or swelling.
# """
minor_response_example = " my brain has exploded"
# --- CHOOSE WHICH TEXT TO TEST ---
# You can switch between the examples or write your own.
# text_to_test = serious_response_example
text_to_test = minor_response_example


# --- 2. RUN THE CLASSIFICATION ---
print("🚀 Starting seriousness classification test...")
print("Analyzing the following text:\n---")
print(text_to_test)
print("---\n")

is_serious = classify_response_seriousness(text_to_test)


# --- 3. DISPLAY THE FINAL RESULT ---
print("="*50)
print("✅ TEST COMPLETE. FINAL OUTPUT:")
print("="*50)

if is_serious:
    print("RESULT: 🔴 SERIOUS (The 'show_hospital_modal' flag would be: True)")
else:
    print("RESULT: 🟢 MINOR (The 'show_hospital_modal' flag would be: False)")

print("="*50)

🚀 Starting seriousness classification test...
Analyzing the following text:
---
 my brain has exploded
---

⚕️ Seriousness check → urgent medical emergency (0.96)
✅ TEST COMPLETE. FINAL OUTPUT:
RESULT: 🔴 SERIOUS (The 'show_hospital_modal' flag would be: True)


In [9]:
!pip install pyngrok


# API SERVER SETUP

In [ ]:
# --- API SERVER SETUP ---

# Install the necessary packages for a web server and tunneling
#!pip install -q fastapi uvicorn pyngrok nest-asyncio

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import uvicorn
import asyncio
from pyngrok import ngrok, conf
import nest_asyncio 
import os
from typing import List

# This is a patch to allow the uvicorn server to run inside the notebook's event loop
nest_asyncio.apply()

#  NGROK AUTHENTICATION 
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
NGROK_AUTH_TOKEN = user_secrets.get_secret("NGROK_AUTH_TOKEN")
conf.get_default().auth_token = NGROK_AUTH_TOKEN

#  DEFINE API DATA STRUCTURES 
class ChatMessage(BaseModel):
    role: str
    content: str

class QueryRequest(BaseModel):
    query: str
    history: List[ChatMessage] = []

class QueryResponse(BaseModel):
    answer: str
    show_hospital_modal: bool # The new flag for the frontend

# Initialize the FastAPI application
app = FastAPI()

# --- THE MAIN API ENDPOINT ---
@app.post("/ask", response_model=QueryResponse)
async def ask_question(request: QueryRequest):
    query = request.query
    history = request.history
    print(f"Received query: '{query}'")

    # 1. Classify the user's intent first
    intent = classify_intent_advanced(query, history)
    print(f"Classified intent as: {intent}")

    try:
        # --- PATH 1: GREETING ---
        if intent == "greeting":
            answer = "Hello! I am a first aid assistant. How can I help you with your medical situation today?"
            return QueryResponse(answer=answer, show_hospital_modal=False)

        # --- PATH 2: OFF-TOPIC ---
        elif intent == "off_topic":
            answer = "I apologize, but I am a specialized medical first aid assistant. I cannot provide information on topics outside of that scope."
            return QueryResponse(answer=answer, show_hospital_modal=False)
            
        # --- PATH 3: MEDICAL QUESTION ---
        elif intent == "medical":
            print("Performing RAG search for medical query...")
            context = query_rag(
                query=query, model=embedding_model, index=index,
                texts=texts, device=device, top_k=5
            )
            
            # Generate the text answer using history
            answer_text = generate_answer(query, context, history)
            
            # Analyze the generated answer for seriousness
            is_serious = classify_response_seriousness(answer_text)
            
            print(f"Generated answer (first 100 chars): {answer_text[:100]}...")
            print(f"Show hospital modal flag set to: {is_serious}")

            # Return the full response object
            return QueryResponse(answer=answer_text, show_hospital_modal=is_serious)

    except Exception as e:
        print(f"An error occurred during generation: {e}")
        raise HTTPException(status_code=500, detail=str(e))


# --- LAUNCH THE SERVER ---         
print("Starting server and creating public tunnel...")    
loop = asyncio.get_event_loop()  
config = uvicorn.Config(app=app, host="0.0.0.0", port=8000, loop=loop)
server = uvicorn.Server(config)
public_url = ngrok.connect(8000)
print(f"Public API is live at: {public_url}")
loop.run_until_complete(server.serve())   

Starting server and creating public tunnel...


INFO:     Started server process [39]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


Public API is live at: NgrokTunnel: "https://randi-interfaith-federico.ngrok-free.dev" -> "http://localhost:8000"
Received query: 'My friend collapsed during a run and isn't responding.'
🧠 Intent classified as MEDICAL (0.86)
Classified intent as: medical
Performing RAG search for medical query...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Both `max_new_tokens` (=1000) and `max_length`(=1000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


⚕️ Seriousness check → urgent medical emergency (0.66)
Generated answer (first 100 chars): You are a reliable, calm, and knowledgeable medical first aid assistant. You can remember the past c...
Show hospital modal flag set to: True
INFO:     41.90.182.36:0 - "POST /ask HTTP/1.1" 200 OK
